In [12]:
import torch


E = 1  #
L = 1.0
I = 1/12
A = 1.0
K = torch.zeros(6, 6) 

import numpy as np

def generate_2d_beam_stiffness(E, A, I, L):

    k = np.zeros((6, 6))
    
    # 轴向刚度项 (EA/L)
    axial = E * A / L
    k[0, 0] = k[3, 3] = axial
    k[0, 3] = k[3, 0] = -axial
    
    # 弯曲和剪切刚度项
    shear = 12 * E * I / L ** 3      # 12EI/L^3
    moment1 = 6 * E * I / L ** 2     # 6EI/L^2
    moment2 = 4 * E * I / L          # 4EI/L (主对角线)
    moment3 = 2 * E * I / L           # 2EI/L (副对角线)
    
    # 填充横向位移和弯曲相关项
    k[1, 1] = k[4, 4] = shear
    k[1, 4] = k[4, 1] = -shear
    
    k[1, 2] = k[2, 1] = k[1, 5] = k[5, 1] = moment1
    k[4, 2] = k[2, 4] = k[4, 5] = k[5, 4] = -moment1
    
    k[2, 2] = k[5, 5] = moment2
    k[2, 5] = k[5, 2] = moment3
    
    k = torch.tensor(k, dtype=torch.float32)
    return k


k_local_1 = generate_2d_beam_stiffness(E, A, I, L)
k_local_2 = generate_2d_beam_stiffness(E, A, I, L)


In [26]:
import torch
import math

# 预先计算所有余弦值
cos_25 = math.cos(0.25 * math.pi)
cos_75 = math.cos(0.75 * math.pi)
cos_neg25 = math.cos(-0.25 * math.pi)
cos_neg75 = math.cos(-0.75 * math.pi)

# 坐标张量
coords = torch.tensor([
    [0.0, 0.0],
    [0.707, 0.707], 
    [1.414, 0.0]
], dtype=torch.float32)

# lambda张量
lamda_1 = torch.tensor([
    [cos_25, cos_75],
    [cos_neg25, cos_25]
], dtype=torch.float32)

lamda_2 = torch.tensor([
    [cos_neg25, cos_25],
    [cos_neg75, cos_neg25]    
], dtype=torch.float32)

# 假设的局部刚度矩阵 (示例值，实际应用中需要真实值)
k_local_1 = torch.rand((6, 6), dtype=torch.float32)
k_local_2 = torch.rand((6, 6), dtype=torch.float32)

# 创建转换矩阵
matrix_T1 = torch.zeros((6, 6), dtype=torch.float32)
matrix_T2 = torch.zeros((6, 6), dtype=torch.float32)

for i in range(0, 6, 2):
    matrix_T1[i:i+2, i:i+2] = lamda_1
    matrix_T2[i:i+2, i:i+2] = lamda_2

# 计算单元刚度矩阵
K_element_1 = torch.matmul(torch.transpose(matrix_T1, 0, 1), 
                          torch.matmul(k_local_1, matrix_T1))
K_element_2 = torch.matmul(torch.transpose(matrix_T2, 0, 1),
                          torch.matmul(k_local_2, matrix_T2))

print("K_element_1 shape:", K_element_1.shape)
print("K_element_2 shape:", K_element_2.shape)

# 组装全局刚度矩阵
# 假设:
# - 单元1连接节点0和1 (对应全局自由度0-3)
# - 单元2连接节点1和2 (对应全局自由度2-5)
# 总自由度数 = 3节点 * 2 = 6

K_global = torch.zeros((6, 6), dtype=torch.float32)

# 组装单元1 (节点0-1: 自由度0-3)
K_global[0:4, 0:4] += K_element_1[0:4, 0:4]  # 只取前4x4部分

# 组装单元2 (节点1-2: 自由度2-5)
K_global[2:6, 2:6] += K_element_2[2:6, 2:6]  # 只取后4x4部分

print("\nAssembled Global Stiffness Matrix:")
print(K_global)


K_element_1 shape: torch.Size([6, 6])
K_element_2 shape: torch.Size([6, 6])

Assembled Global Stiffness Matrix:
tensor([[ 0.9490, -0.6536,  1.3998, -0.1279,  0.0000,  0.0000],
        [ 0.2560, -0.1058,  0.0760,  0.1647,  0.0000,  0.0000],
        [ 0.9256, -0.2783,  1.2912,  0.3563, -0.1642,  0.4329],
        [-0.1559,  0.0154,  0.0142,  0.7590, -0.1315,  0.5629],
        [ 0.0000,  0.0000,  0.2114,  0.5738, -0.2582, -0.3995],
        [ 0.0000,  0.0000, -0.1968,  1.1441,  0.1794,  0.9112]])


In [29]:
F = torch.tensor([0.0, 0.0, 0.0, -1.0, 0.0, 0.0])
i = torch.tensor([0,1,4,5])
for g in i :
    K_global[g, :] = 0
    K_global[:, g] = 0
    K_global[g, g] = 1

U = torch.linalg.solve(K_global, F)